# 30. Dynamic Shape Handling | 动态 Shape 处理

**难度：** Medium-Hard | **环境：** CPU-first | **标签：** `系统工程`, `动态 Shape`, `推理服务` | **目标人群：** 编译与系统入门者

![动态形状、分桶与执行路径](../docs/public/01_Hardware_Math_and_Systems/30_dynamic_shape_specialization.svg)

---

## 本节导读

输入长度一旦不固定，问题就不再只是 padding 多一点少一点，而是 batching、缓存组织、kernel 选择和执行路径都会一起变复杂。静态 shape 下顺手成立的很多假设，到了动态输入场景里都可能失效。

本节沿着形状变化、编译路径复用、分桶填充与执行分支建立动态输入的系统直觉，为推理调度和算子优化提供共同基础。学习重点是区分形状多样性带来的额外计算、缓存失配与路径切换。

**关键词：** `dynamic shape`, `batching`, `padding`

---

## 前置阅读

**导语：** 这一页先把动态 batching、缓存复用和执行路径的关系接上，再看输入长度变化为什么会让系统更难稳定优化。

- [07. CPU and GPU Heterogeneous Scheduling | CPU 与 GPU 异构调度](./07_CPU_GPU_Heterogeneous_Scheduling.ipynb)
- [08. Programming Models and CUDA/Triton | 编程模型演进](./08_Programming_Models_CUDA_Triton.ipynb)
- [29. CUDA Stream Advanced Scheduling | CUDA Stream 高级调度](./29_CUDA_Stream_Advanced_Scheduling.ipynb)


## Q1：动态 shape 为什么会让复用变差？


固定 shape 时，batch 形状、缓存布局、kernel 形态和 graph 回放都更容易复用。

一旦输入长度不固定，这些复用都会变差：不同请求要重新分桶，padding 会引入无效计算，graph / kernel 复用率下降，执行路径也更容易分叉。

所以动态 shape 的难点，不是长度变化本身，而是它让原本能重复利用的优化结果变得不稳定。


### Q1小验证：固定假设为什么会失效

先记住固定 shape 下哪些结果能复用，再看动态输入会打断哪些复用。

In [ ]:
def shape_reuse_report(shapes):
    """按实际 shape 序列统计编译路径数、缓存命中数和复用率。"""
    if not shapes or any(not shape for shape in shapes):
        raise ValueError('shapes must be a non-empty sequence of non-empty tuples')
    normalized = [tuple(shape) for shape in shapes]
    unique_shapes = set(normalized)
    cache_misses = len(unique_shapes)
    cache_hits = len(normalized) - cache_misses
    return {'requests': len(normalized), 'compiled_paths': cache_misses, 'cache_hits': cache_hits, 'reuse_ratio': cache_hits / len(normalized)}

fixed = shape_reuse_report([(8, 512)] * 4)
dynamic = shape_reuse_report([(8, 128), (8, 512), (8, 128), (8, 2048)])
assert fixed['reuse_ratio'] > dynamic['reuse_ratio']
print('fixed:', fixed)
print('dynamic:', dynamic)


## Q2：padding 和 bucketing 分别在减少什么浪费？


- **padding** 减少的是形状不一致带来的拼 batch 难题，它用填充把不同长度的输入拼成更规则的张量；
- **bucketing** 减少的是无效 padding，它把相近长度的请求放到同一桶里，让浪费更少。

两者的目标都不是“消灭长度差异”，而是在吞吐、显存浪费和 kernel 稳定性之间找折中。

如果没有这层处理，动态 batching 很容易同时损失计算效率和资源利用率。


### Q2小验证：什么时候更适合 bucketing

长度分布越分散，bucketing 对减少 padding 浪费越有价值。

In [ ]:
def padding_overhead(lengths, bucket_boundaries=None):
    """计算单一大 Batch 或按长度边界分桶后的填充 Token 数。"""
    if not lengths or any(length <= 0 for length in lengths):
        raise ValueError('lengths must be positive and non-empty')
    if bucket_boundaries is None:
        targets = [max(lengths)] * len(lengths)
    else:
        boundaries = sorted(set(bucket_boundaries))
        if not boundaries or boundaries[-1] < max(lengths) or any(boundary <= 0 for boundary in boundaries):
            raise ValueError('bucket boundaries must cover every length')
        targets = [next(boundary for boundary in boundaries if length <= boundary) for length in lengths]
    wasted_tokens = sum(target - length for target, length in zip(targets, lengths))
    return {'wasted_tokens': wasted_tokens, 'total_tokens': sum(targets), 'padding_ratio': wasted_tokens / sum(targets)}

lengths = [64, 128, 256, 520, 800]
single_batch = padding_overhead(lengths)
bucketed = padding_overhead(lengths, [128, 256, 512, 1024])
assert bucketed['wasted_tokens'] < single_batch['wasted_tokens']
print('single batch:', single_batch)
print('bucketed:', bucketed)


## Q3：动态 shape 为什么会让性能更容易抖动？


动态 shape 会同时影响两层决策：

1. **执行路径**：不同长度可能触发不同 kernel、不同 graph 或不同调度分支。
2. **缓存策略**：原本在固定形状下可复用的编译结果、块大小、缓存布局和调度参数，在动态场景里可能要按形状重新选择。

所以动态 shape 的本质，是把“编译期可确定的执行链”变成“依赖 runtime 形状判断的执行链”，性能也就更容易抖动。


### Q3小验证：为什么性能会抖动

长度变化导致路径和缓存都跟着变时，性能就更容易抖。

In [ ]:
def path_stability(lengths, boundaries=(256, 1024)):
    """统计运行时路径种类和相邻请求的路径切换次数。"""
    if not lengths or any(length <= 0 for length in lengths):
        raise ValueError('lengths must be positive and non-empty')
    if len(boundaries) != 2 or not 0 < boundaries[0] < boundaries[1]:
        raise ValueError('boundaries must be two increasing positive values')
    def route(length):
        return 'short' if length < boundaries[0] else 'long' if length < boundaries[1] else 'very_long'
    paths = [route(length) for length in lengths]
    transitions = sum(left != right for left, right in zip(paths, paths[1:]))
    return {'paths': paths, 'unique_paths': len(set(paths)), 'path_transitions': transitions, 'stable': transitions == 0}

stable = path_stability([128, 128, 128])
mixed = path_stability([128, 512, 128, 2048])
assert stable['stable'] is True
assert mixed['unique_paths'] == 3 and mixed['path_transitions'] == 3
print('stable:', stable)
print('mixed:', mixed)


## ⚠️ 常见误区

- 动态 shape 不是“只是输入长度不同”。
- padding 能解决对齐，但不等于没有浪费。
- bucketing 不是万能，只是减少浪费的办法之一。
- 动态输入场景里，执行路径和缓存复用都可能变差。

## 相关阅读

**导语：** 如果想继续把动态输入和后续的调度、虚拟化、成本判断串起来，可以接着看这些页。

- [19. Operator Fusion Introduction | 算子融合导论](./19_Operator_Fusion_Introduction.ipynb)
- [15. CUDA Custom Kernel Intro | CUDA 自定义算子入门](../04_CUDA_and_System_Optimization/15_CUDA_Custom_Kernel_Intro.ipynb)
- [18. CUDA Graph and JIT Compile | CUDA Graph 与 JIT 编译](../04_CUDA_and_System_Optimization/18_CUDA_Graph_and_JIT_Compile.ipynb)
---